# Preprocessing — Scoring Credit Ivoirien
**Objectif :** Preparer les donnees pour le reseau de neurones.

In [6]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print("Bibliotheques importees !")

Bibliotheques importees !


In [7]:
# Chargement
TAUX_USD_FCFA = 600

df = pd.read_csv(
    r'C:\Users\emman\Documents\Portfolio\credit-scoring-deep-learning\data\cs-training.csv'
)
df = df.drop(columns=['Unnamed: 0'])
df['MonthlyIncome_FCFA'] = df['MonthlyIncome'] * TAUX_USD_FCFA

print(f"Dataset charge : {len(df):,} lignes")
print(f"Valeurs manquantes :\n{df.isnull().sum()}")

Dataset charge : 150,000 lignes
Valeurs manquantes :
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
MonthlyIncome_FCFA                      29731
dtype: int64


In [8]:
# Nettoyage des valeurs aberrantes
print("Avant nettoyage :")
print(f"  Age min      : {df['age'].min()}")
print(f"  Age max      : {df['age'].max()}")
print(f"  Revenu max   : {df['MonthlyIncome_FCFA'].max():,.0f} FCFA")

# Corriger les ages aberrants
df = df[df['age'] > 0]

# Plafonner les revenus aberrants
# Max realiste en CI : 50 000 USD = 30 000 000 FCFA
df['MonthlyIncome_FCFA'] = df['MonthlyIncome_FCFA'].clip(
    upper=30_000_000
)
df['MonthlyIncome'] = df['MonthlyIncome'].clip(
    upper=50_000
)

# Plafonner les ratios aberrants
df['RevolvingUtilizationOfUnsecuredLines'] = df[
    'RevolvingUtilizationOfUnsecuredLines'
].clip(upper=1)

df['DebtRatio'] = df['DebtRatio'].clip(upper=10)

print("\nApres nettoyage :")
print(f"  Lignes restantes : {len(df):,}")
print(f"  Age min          : {df['age'].min()}")
print(f"  Age max          : {df['age'].max()}")
print(f"  Revenu max       : {df['MonthlyIncome_FCFA'].max():,.0f} FCFA")

Avant nettoyage :
  Age min      : 0
  Age max      : 109
  Revenu max   : 1,805,250,000 FCFA

Apres nettoyage :
  Lignes restantes : 149,999
  Age min          : 21
  Age max          : 109
  Revenu max       : 30,000,000 FCFA


In [9]:
# Imputation des valeurs manquantes
imputer_median = SimpleImputer(strategy='median')

# MonthlyIncome → imputer par la mediane
df['MonthlyIncome'] = imputer_median.fit_transform(
    df[['MonthlyIncome']]
)
df['MonthlyIncome_FCFA'] = df['MonthlyIncome'] * TAUX_USD_FCFA

# NumberOfDependents → imputer par la mediane
df['NumberOfDependents'] = imputer_median.fit_transform(
    df[['NumberOfDependents']]
)

print("Imputation terminee !")
print(f"Valeurs manquantes restantes :")
print(df.isnull().sum())

Imputation terminee !
Valeurs manquantes restantes :
SeriousDlqin2yrs                        0
RevolvingUtilizationOfUnsecuredLines    0
age                                     0
NumberOfTime30-59DaysPastDueNotWorse    0
DebtRatio                               0
MonthlyIncome                           0
NumberOfOpenCreditLinesAndLoans         0
NumberOfTimes90DaysLate                 0
NumberRealEstateLoansOrLines            0
NumberOfTime60-89DaysPastDueNotWorse    0
NumberOfDependents                      0
MonthlyIncome_FCFA                      0
dtype: int64


In [10]:
# Feature Engineering — nouvelles variables metier
# Contexte ivoirien

# 1. Categorie de revenu FCFA
def categoriser_revenu(revenu_fcfa):
    if revenu_fcfa < 75_000:
        return 0  # En dessous du SMIC
    elif revenu_fcfa < 225_000:
        return 1  # Revenu modeste
    elif revenu_fcfa < 700_000:
        return 2  # Revenu moyen
    else:
        return 3  # Revenu eleve

df['categorie_revenu'] = df['MonthlyIncome_FCFA'].apply(
    categoriser_revenu
)

# 2. Total des retards de paiement
df['total_retards'] = (
    df['NumberOfTime30-59DaysPastDueNotWorse'] +
    df['NumberOfTime60-89DaysPastDueNotWorse'] +
    df['NumberOfTimes90DaysLate']
)

# 3. Client a risque (retards graves)
df['client_risque'] = (
    df['NumberOfTimes90DaysLate'] > 0
).astype(int)

# 4. Ratio revenu/dependants
df['revenu_par_dependant'] = df['MonthlyIncome_FCFA'] / (
    df['NumberOfDependents'] + 1
)

print("Feature Engineering termine !")
print(f"\nNouvelles variables creees :")
print(f"  categorie_revenu        : {df['categorie_revenu'].value_counts().to_dict()}")
print(f"  total_retards moyen     : {df['total_retards'].mean():.3f}")
print(f"  clients a risque        : {df['client_risque'].sum():,} ({df['client_risque'].mean()*100:.2f}%)")
print(f"  revenu_par_dependant    : {df['revenu_par_dependant'].mean():,.0f} FCFA")

Feature Engineering termine !

Nouvelles variables creees :
  categorie_revenu        : {3: 144721, 2: 2753, 0: 2307, 1: 218}
  total_retards moyen     : 0.927
  clients a risque        : 8,338 (5.56%)
  revenu_par_dependant    : 2,696,537 FCFA


In [11]:
# Separation X et y
features = [
    'RevolvingUtilizationOfUnsecuredLines',
    'age',
    'NumberOfTime30-59DaysPastDueNotWorse',
    'DebtRatio',
    'MonthlyIncome_FCFA',
    'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate',
    'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse',
    'NumberOfDependents',
    'categorie_revenu',
    'total_retards',
    'client_risque',
    'revenu_par_dependant'
]

X = df[features]
y = df['SeriousDlqin2yrs']

# Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train : {len(X_train):,} lignes")
print(f"Test  : {len(X_test):,} lignes")
print(f"\nDefauts dans train : {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Defauts dans test  : {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

# Normalisation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nNormalisation effectuee !")
print(f"Shape X_train : {X_train_scaled.shape}")
print(f"Shape X_test  : {X_test_scaled.shape}")

Train : 119,999 lignes
Test  : 30,000 lignes

Defauts dans train : 8,021 (6.68%)
Defauts dans test  : 2,005 (6.68%)

Normalisation effectuee !
Shape X_train : (119999, 14)
Shape X_test  : (30000, 14)


In [12]:
# Sauvegarder pour le notebook Deep Learning
import numpy as np

np.save(
    r'C:\Users\emman\Documents\Portfolio\credit-scoring-deep-learning\data\X_train.npy',
    X_train_scaled
)
np.save(
    r'C:\Users\emman\Documents\Portfolio\credit-scoring-deep-learning\data\X_test.npy',
    X_test_scaled
)
np.save(
    r'C:\Users\emman\Documents\Portfolio\credit-scoring-deep-learning\data\y_train.npy',
    y_train.values
)
np.save(
    r'C:\Users\emman\Documents\Portfolio\credit-scoring-deep-learning\data\y_test.npy',
    y_test.values
)

# Sauvegarder les noms des features
import json
with open(
    r'C:\Users\emman\Documents\Portfolio\credit-scoring-deep-learning\data\features.json',
    'w'
) as f:
    json.dump(features, f)

print("Donnees sauvegardees !")
print(f"  X_train.npy : {X_train_scaled.shape}")
print(f"  X_test.npy  : {X_test_scaled.shape}")
print(f"  y_train.npy : {y_train.shape}")
print(f"  y_test.npy  : {y_test.shape}")
print(f"  features.json : {len(features)} features")

Donnees sauvegardees !
  X_train.npy : (119999, 14)
  X_test.npy  : (30000, 14)
  y_train.npy : (119999,)
  y_test.npy  : (30000,)
  features.json : 14 features
